# 08 - Qwen Briefs & Rubric Evaluation (Local)

**Insurance Claim Fraud Detection & Action Recommendation System**

This notebook runs entirely **locally** - conda/Jupyter plus a local Ollama
server running `qwen2.5:3b`. It needs nothing from Notebooks 01-07 except
one file, `shap_claim_briefs.json`, copied into this notebook's working
directory. No Google Drive, no fraud model, no preprocessing pipeline, no
SHAP, no cloud API.

**What this notebook does:**
1. Step 0 - confirms Ollama is reachable and `qwen2.5:3b` is pulled
2. Step 1 - builds a factual **skeleton in code** per claim (features already
   sorted into "increased fraud risk" / "reduced fraud risk" by their SHAP
   direction, action already fixed by risk tier), then a prompt that hands
   Qwen that finished skeleton from a git-ignored template
   (`prompts/brief_prompt.txt`)
3. Step 2 - Qwen **renders** the skeleton into a 3-4 sentence investigation
   brief - it does not sort features or choose the action, only writes
   prose - with per-claim retry, saved to `results/generated_briefs.json`
   alongside the code-built skeleton itself
4. Step 3 - builds `rubric/brief_scores.csv`: each brief next to its SHAP
   drivers, with blank columns for a human to score 0/1/2 on four criteria,
   plus a cheap automated fabrication pre-check
5. Step 4 - once a human has filled in the rubric, computes the gate pass
   rate, mean score per criterion, and surfaces example briefs for the report

**Explicitly out of scope here:** the fraud model, the preprocessing
pipeline, SHAP, and anything Colab/Drive-related. This notebook cannot
re-score a claim or re-run the model, and it does not decide whether a
claim actually is fraud - it only explains, in natural language, why the
already-frozen model already flagged it, using SHAP's own stated reasons,
and gets that explanation checked.

**Honest scope, specific to this notebook:** `qwen2.5:3b` is a small local
model. Earlier versions of this notebook let Qwen decide which features
increased vs. reduced fraud risk itself, and it flipped that assignment on
roughly a quarter of claims - a small model's ceiling on this task, not
something more prompt engineering could fix. That failure mode is now
structurally impossible: the code sorts every feature by its SHAP
`direction` field and fixes the action from the risk tier *before* Qwen
ever sees the claim, and Qwen is only asked to render that finished
skeleton into prose, never to sort or decide anything (see the "Why this
generalizes" note in Section 3). What Qwen can still get wrong - and why
Step 3 still exists - is a fabricated detail, or prose that reads fluently
but drifts from the given skeleton in some other way. Every brief is still
meant to be checked against its own skeleton before anyone treats it as
ground truth.

**Assumes this notebook's working directory is wherever you launch Jupyter
for it** (typically the `notebooks/` folder) - every path below
(`shap_claim_briefs.json`, `prompts/`, `results/`, `rubric/`) is relative to
that, not to Google Drive or a fixed repo root.

## 1. Environment Setup

Unlike Notebooks 01-07, there is no Google Drive, no `PROJECT_ROOT`, and no
GPU model here - just local paths and one HTTP client (`requests`).
`random_state` isn't meaningful in this notebook either: Notebook 07 already
fixed which ~30 claims exist and in what order; nothing here samples
anything new.

In [43]:
"""Notebook 08 - Qwen Briefs & Rubric Evaluation: imports and shared configuration."""
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
!pip install pandas
import pandas as pd
import requests
from IPython.display import display

OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen2.5:3b"
# Greedy decoding, deliberately - NOT Qwen's general-purpose recommended defaults.
# temperature=0.7/top_p=0.8/top_k=20 and repeat_penalty>1 were tried here to fix an
# earlier bug (the model copying fallback boilerplate it saw in-context), but that fix
# is now handled structurally (Section 3: the fallback text is never shown to the model
# at all, regardless of temperature) - so there's no remaining reason to trade away
# faithfulness for creative-sampling defaults tuned for open-ended chat. This task is
# narrow ("restate these exact given facts, add nothing"), and creative sampling
# actively works against it: repeat_penalty in particular penalizes the model for
# literally repeating tokens it just saw in its own short prompt (Ollama/llama.cpp's
# repeat window includes prompt context), nudging it toward paraphrase and invented
# rationalization instead of verbatim restatement - exactly the failure mode observed
# after the earlier switch to creative sampling. seed is kept for explicitness, though
# at temperature=0 greedy decoding is already fully deterministic on its own.
GENERATION_OPTIONS = {
    "temperature": 0,
    "repeat_penalty": 1.0,
    "seed": 42,
}
MAX_RETRIES = 1  # one retry after a transport failure, per Ollama call

# The two sentences for an empty driver group are always composed by CODE, never
# requested from or shown to the model (see Section 3's structural fix) - this is their
# single source of truth, referenced both when splicing a final brief together and when
# asserting the prompt template never leaks this literal text to the model.
FIXED_SENTENCES = {
    "increased": "No factors increased the fraud risk.",
    "reduced": "No factors reduced the fraud risk.",
}

SHAP_EXPORT_PATH = Path("shap_claim_briefs.json")        # Notebook 07's handoff file, copied here by hand
PROMPT_TEMPLATE_PATH = Path("prompts/brief_prompt.txt")  # git-ignored - see Section 3
GENERATED_BRIEFS_PATH = Path("results/generated_briefs.json")
RUBRIC_SCORES_PATH = Path("rubric/brief_scores.csv")

RUBRIC_CRITERIA = ["factual_consistency", "faithfulness_to_shap", "actionability", "no_fabrication"]

print("Configuration")
print("-" * 60)
print(f"OLLAMA_BASE_URL       : {OLLAMA_BASE_URL}")
print(f"MODEL_NAME            : {MODEL_NAME}")
print(f"GENERATION_OPTIONS    : {GENERATION_OPTIONS}")
print(f"SHAP_EXPORT_PATH      : {SHAP_EXPORT_PATH.resolve()}")
print(f"PROMPT_TEMPLATE_PATH  : {PROMPT_TEMPLATE_PATH.resolve()}")
print(f"GENERATED_BRIEFS_PATH : {GENERATED_BRIEFS_PATH.resolve()}")
print(f"RUBRIC_SCORES_PATH    : {RUBRIC_SCORES_PATH.resolve()}")


Configuration
------------------------------------------------------------
OLLAMA_BASE_URL       : http://localhost:11434
MODEL_NAME            : qwen2.5:3b
GENERATION_OPTIONS    : {'temperature': 0, 'repeat_penalty': 1.0, 'seed': 42}
SHAP_EXPORT_PATH      : C:\Users\LENOVO\Desktop\samsung-project\insurance-fraud-detection\notebooks\shap_claim_briefs.json
PROMPT_TEMPLATE_PATH  : C:\Users\LENOVO\Desktop\samsung-project\insurance-fraud-detection\notebooks\prompts\brief_prompt.txt
GENERATED_BRIEFS_PATH : C:\Users\LENOVO\Desktop\samsung-project\insurance-fraud-detection\notebooks\results\generated_briefs.json
RUBRIC_SCORES_PATH    : C:\Users\LENOVO\Desktop\samsung-project\insurance-fraud-detection\notebooks\rubric\brief_scores.csv


## 2. Step 0 - Connectivity Check

Two things must both be true before anything else in this notebook makes
sense: Ollama itself must be reachable, and the specific model
(`qwen2.5:3b`) must already be pulled. Failing loudly here, with the exact
fix, beats a confusing timeout ten cells later.

In [44]:
def check_ollama_available(base_url: str, model_name: str) -> None:
    """Fail loudly with a clear, actionable message if Ollama isn't reachable or the
    model isn't pulled - the whole notebook depends on both being true."""
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=5)
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        raise RuntimeError(
            f"Could not reach the Ollama server at {base_url}.\n\n"
            "This notebook needs a local Ollama server - nothing here calls a cloud "
            "API. To fix:\n"
            "  1. Install Ollama from https://ollama.com if you haven't already.\n"
            "  2. Start it (it usually runs as a background service; otherwise run "
            "`ollama serve` in a terminal).\n"
            f"  3. Confirm it's listening at {base_url} (try `curl {base_url}/api/tags`).\n"
            "  4. Re-run this cell."
        ) from exc

    models = response.json().get("models", [])
    available_names = set()
    for m in models:
        for key in ("name", "model"):
            if m.get(key):
                available_names.add(m[key])

    model_available = model_name in available_names or any(
        name.startswith(f"{model_name}:") for name in available_names
    )
    if not model_available:
        raise RuntimeError(
            f"Ollama is reachable, but '{model_name}' is not pulled.\n\n"
            f"Models currently available: {sorted(available_names) or '(none)'}\n\n"
            f"Fix: run `ollama pull {model_name}` in a terminal, then re-run this cell."
        )
    print(f"PASSED: Ollama is reachable at {base_url}, and '{model_name}' is available.")


check_ollama_available(OLLAMA_BASE_URL, MODEL_NAME)

PASSED: Ollama is reachable at http://localhost:11434, and 'qwen2.5:3b' is available.


### Load Notebook 07's handoff file

`shap_claim_briefs.json` is the *only* input this notebook needs beyond
Ollama itself - read-only, never modified. Its structure is validated here
so a malformed or wrong file fails loudly now, not silently mid-generation.

In [45]:
if not SHAP_EXPORT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {SHAP_EXPORT_PATH.resolve()}.\n\n"
        "This notebook needs Notebook 07's handoff export. Copy "
        "'shap_claim_briefs.json' from the Colab Drive results/ folder into this "
        "notebook's working directory, then re-run this cell."
    )

with open(SHAP_EXPORT_PATH, "r", encoding="utf-8") as f:
    shap_export = json.load(f)

# Notebook 07 wraps claims under a "claims" key alongside run metadata; accept a bare
# list too, so this notebook isn't brittle to that one structural choice.
claims = shap_export["claims"] if isinstance(shap_export, dict) else shap_export

REQUIRED_CLAIM_FIELDS = {"claim_id", "calibrated_fraud_probability", "risk_tier",
                          "true_label", "prediction_outcome", "top_shap_drivers",
                          "original_feature_values"}
missing_field_claims = [c.get("claim_id", "<no id>") for c in claims if not REQUIRED_CLAIM_FIELDS.issubset(c)]
if missing_field_claims:
    raise ValueError(
        f"{len(missing_field_claims)} claim(s) are missing required field(s) - expected "
        f"{sorted(REQUIRED_CLAIM_FIELDS)} on every record. First affected: "
        f"{missing_field_claims[:5]}. Is this really Notebook 07's export?"
    )

claims_by_id = {c["claim_id"]: c for c in claims}
print(f"Loaded {len(claims)} claims from {SHAP_EXPORT_PATH.name}.")
print(f"Outcome mix: {pd.Series([c['prediction_outcome'] for c in claims]).value_counts().to_dict()}")

Loaded 46 claims from shap_claim_briefs.json.
Outcome mix: {'false_positive': 15, 'true_positive': 15, 'true_negative': 14, 'false_negative': 2}


## 3. Step 1 - Skeleton + Prompt Construction

**The direction and action assignment happens here, in code - not in the prompt, and
not by the model.** For each claim: `split_drivers_by_direction()` splits its top SHAP
drivers (already trimmed to `TOP_N_DRIVERS_FOR_PROMPT`) into an "increased fraud risk"
group and a "reduced fraud risk" group purely from each driver's own `direction` field,
and `derive_action_from_tier()` looks the action up from a fixed `risk_tier -> action`
mapping. Both are finished before any prompt is built. Qwen is never shown an
unsorted list and never asked to classify a feature or pick an action - it only
renders the two already-sorted groups and the already-fixed action into prose.

The actual prompt text lives in `prompts/brief_prompt.txt`, **git-ignored** - not
because it's secret, but so the exact wording can be iterated on locally without every
tweak becoming a commit, and so this repo doesn't carry prompt text that's really a
local experimentation artifact. If that file doesn't exist yet (true on a fresh
checkout, since it's git-ignored), the next cell writes a starter version for you and
never touches it again.

**Grounding design:** the skeleton's two feature groups are exactly this claim's top
`TOP_N_DRIVERS_FOR_PROMPT` SHAP drivers - the model's own stated reasons for the flag -
not a separately curated list from the other ~20 raw columns on the claim. This keeps
what the LLM is given, and what Step 3's fabrication check treats as "in scope," the
same bounded vocabulary.

In [46]:
DEFAULT_PROMPT_TEMPLATE = """[SYSTEM]
You are a technical writer producing one factual sentence at a time for an official fraud-review brief. You will be given a short list of already-finalized facts and asked to turn ONLY those facts into a single plain-prose sentence. You are not investigating, sorting, or deciding anything - every fact you are given is already final and was placed there by code, not by you.

RULES:
- Use only the facts given to you in this message. Do not add, omit, rename, or reinterpret any feature name or value.
- Every factor given to you is present and already confirmed to increase or lower the fraud risk - state it plainly. Never say a factor "does not contribute," "has no impact," "had no specified impact," "is not present," "did not apply," or "is missing."
- State only that a factor increased or lowered the risk - do not explain, justify, or speculate about why, and do not add any interpretation, causal reasoning, or context not given to you in this message.
- "Year" refers to the claim year, never vehicle age.
- Write one plain-prose sentence only - no bullet points, no headers, no meta-commentary about your task, no mention of any other claim or any other case.

Respond in exactly this format, nothing else:
BRIEF: <your one sentence>

[USER]
Write one sentence using only the exact facts stated below:

{directives}
"""

if not PROMPT_TEMPLATE_PATH.exists():
    PROMPT_TEMPLATE_PATH.parent.mkdir(parents=True, exist_ok=True)
    PROMPT_TEMPLATE_PATH.write_text(DEFAULT_PROMPT_TEMPLATE, encoding="utf-8")
    print(f"'{PROMPT_TEMPLATE_PATH}' did not exist (expected - it's git-ignored) - wrote "
          "a starter template. Edit it freely; this notebook only writes it once and "
          "never overwrites an existing copy.")

raw_template = PROMPT_TEMPLATE_PATH.read_text(encoding="utf-8")

# Two sections, not one flat template: the model must never see a single
# undifferentiated block mixing durable rules with per-claim facts. Ollama's /api/chat
# separates these into distinct roles, matching how Qwen2.5 was instruction-tuned
# (ChatML), rather than relying on the model to infer the split itself from one string.
assert "[SYSTEM]" in raw_template and "[USER]" in raw_template, (
    f"'{PROMPT_TEMPLATE_PATH}' must contain both a '[SYSTEM]' and a '[USER]' section "
    "marker - restore it (or delete the file to regenerate the starter template) "
    "before proceeding."
)
system_part, _, user_part = raw_template.partition("[USER]")
system_prompt_template = system_part.replace("[SYSTEM]", "", 1).strip()
user_prompt_template = user_part.strip()

assert "{" not in system_prompt_template and "}" not in system_prompt_template, (
    f"'{PROMPT_TEMPLATE_PATH}' [SYSTEM] section must contain no placeholders - it is "
    "static across every call, never filled in per-claim. Move any per-claim text "
    "into the [USER] section instead."
)
assert "{directives}" in user_prompt_template, (
    f"'{PROMPT_TEMPLATE_PATH}' [USER] section is missing the {{directives}} "
    "placeholder this notebook fills in per group per claim - restore it (or delete "
    "the file to regenerate the starter template) before proceeding."
)

# Load-bearing, not decorative: this is what makes "the model never sees the fallback
# text" enforceable even against a future hand-edit of this git-ignored file. If either
# fixed sentence appears anywhere in the template, the branch-resolution this whole fix
# depends on has been undone - fail loudly now, not via a bad brief later.
for direction, fixed_sentence in FIXED_SENTENCES.items():
    assert fixed_sentence not in raw_template, (
        f"'{PROMPT_TEMPLATE_PATH}' contains the literal fixed '{direction}' sentence "
        f"'{fixed_sentence}' - this sentence must only ever be composed by code "
        "(FIXED_SENTENCES) and spliced into the final brief, never shown to or "
        "requested from the model. Remove it from the template."
    )

print(f"Loaded prompt template from {PROMPT_TEMPLATE_PATH} "
      f"(system: {len(system_prompt_template)} chars, user: {len(user_prompt_template)} chars).")


Loaded prompt template from prompts\brief_prompt.txt (system: 1246 chars, user: 73 chars).


In [ ]:
TOP_N_DRIVERS_FOR_PROMPT = 5  # cap on how many SHAP drivers are shown to the model per claim

ACTION_BY_RISK_TIER = {
    "High": "Fraud Investigation",
    "Medium": "Manual Review",
    "Low": "Approve",
}


def get_shap_contribution(driver: dict) -> float:
    """Notebook 07 exports this as 'shap_contribution'; tolerate 'shap_value' too, in
    case a differently-named export is ever handed to this notebook instead."""
    return float(driver.get("shap_contribution", driver.get("shap_value", 0.0)))


def select_top_drivers(drivers: list, n: int) -> list:
    """Keep only the n drivers with the largest absolute SHAP contribution. Notebook 07
    exports up to 8 per claim, and the bottom few are typically near-zero (+/-0.03) -
    they contribute almost nothing to the actual decision but give the model more
    features to potentially misattribute or invent a connection for."""
    return sorted(drivers, key=lambda d: abs(get_shap_contribution(d)), reverse=True)[:n]


def split_drivers_by_direction(drivers: list) -> tuple:
    """Sort this claim's SHAP drivers into two direction-pure groups *here, in code* -
    not in the prompt, and not by the model. The direction of every driver is a fact
    SHAP already computed (its sign); the model never sees an unsorted list and is
    never asked to classify a feature, so it cannot flip one."""
    increased = [d for d in drivers if d["direction"] == "up"]
    reduced = [d for d in drivers if d["direction"] == "down"]
    unrecognized = [d for d in drivers if d["direction"] not in ("up", "down")]
    if unrecognized:
        raise ValueError(
            "Driver(s) with an unrecognized 'direction' value (expected 'up' or "
            f"'down'): {[(d['feature'], d.get('direction')) for d in unrecognized]}"
        )
    return increased, reduced


def derive_action_from_tier(risk_tier: str) -> str:
    """The action is fixed by the risk tier via this static mapping - decided here, in
    code, never chosen by the model."""
    if risk_tier not in ACTION_BY_RISK_TIER:
        raise ValueError(
            f"Unrecognized risk_tier '{risk_tier}' - expected one of {sorted(ACTION_BY_RISK_TIER)}."
        )
    return ACTION_BY_RISK_TIER[risk_tier]


def format_skeleton_group(drivers: list) -> str:
    """Render one already direction-pure, non-empty group as 'feature = value' pairs.
    Only ever called on a non-empty group - build_group_directive() below returns None
    for an empty one before this is reached, so there is no 'none' case to render here."""
    return "; ".join(f"{d['feature']} = {d['value']}" for d in drivers)


def build_skeleton(claim: dict) -> dict:
    """Build the factual skeleton entirely in code, from THIS claim's own top SHAP
    drivers (already trimmed to TOP_N_DRIVERS_FOR_PROMPT) and risk tier - both computed
    fresh, at inference time, for this specific claim, not looked up from any stored or
    training-time table. Every direction and the action are decided here; nothing
    downstream (the prompt, the model, or the saved record) ever re-derives them - see
    the "Why this generalizes to unseen data" note below."""
    top_drivers = select_top_drivers(claim["top_shap_drivers"], TOP_N_DRIVERS_FOR_PROMPT)
    increased, reduced = split_drivers_by_direction(top_drivers)
    return {
        "increased_fraud_risk": increased,
        "reduced_fraud_risk": reduced,
        "risk_tier": claim["risk_tier"],
        "calibrated_fraud_probability": claim["calibrated_fraud_probability"],
        "action": derive_action_from_tier(claim["risk_tier"]),
    }


def compose_lead_in_sentence(skeleton: dict) -> str:
    """Always code, never the model - states the tier and probability, neither of
    which is ever in question by the time this runs."""
    return (f"This claim is {skeleton['risk_tier']} risk, with a calibrated fraud "
            f"probability of {skeleton['calibrated_fraud_probability']:.2f}.")


def compose_action_sentence(skeleton: dict) -> str:
    """Always code, never the model - the action was already fixed by risk tier in
    build_skeleton(), so there is nothing left for the model to decide or echo."""
    return f"This claim is routed for {skeleton['action']}."


def build_group_directive(direction: str, drivers: list):
    """The per-claim, per-group fact block the model is shown for this group - or None
    if the group is empty, meaning the model is not called for it at all (its sentence
    comes from FIXED_SENTENCES instead). This is the structural fix: the model is never
    shown the OTHER branch's fallback text, because it is never shown anything about an
    empty group in the first place - there is no conditional for it to get wrong."""
    if not drivers:
        return None
    verb = "increased" if direction == "increased" else "lowered"
    return f"The following factors {verb} the fraud risk: {format_skeleton_group(drivers)}."


def assert_no_fallback_leakage(system_text: str, user_text: str) -> None:
    """Runtime counterpart to the template-parse-time assertion in the previous cell -
    re-checked immediately before every single Ollama call, on the exact text about to
    be sent, not just the template it was built from. Catches any code path that could
    reintroduce the fallback text into a live prompt, not only a hand-edited file."""
    combined = f"{system_text}\n{user_text}"
    for direction, fixed_sentence in FIXED_SENTENCES.items():
        assert fixed_sentence not in combined, (
            f"About to send the literal fixed '{direction}' sentence "
            f"'{fixed_sentence}' to the model - this must only ever be spliced into "
            "the final brief by code, never sent as part of a prompt. Aborting this "
            "call rather than risk reproducing the boilerplate-contradiction bug."
        )


example_skeleton = build_skeleton(claims[0])
example_lead_in = compose_lead_in_sentence(example_skeleton)
example_action_sentence = compose_action_sentence(example_skeleton)
example_increased_directive = build_group_directive("increased", example_skeleton["increased_fraud_risk"])
example_reduced_directive = build_group_directive("reduced", example_skeleton["reduced_fraud_risk"])
print(f"Example skeleton for claim {claims[0]['claim_id']}: {example_skeleton}\n")
print(f"Lead-in sentence (code, no LLM): {example_lead_in}")
print(f"Action sentence (code, no LLM): {example_action_sentence}")
print(f"Increased-group directive (None means: skip the model, use "
      f"FIXED_SENTENCES['increased'] instead): {example_increased_directive}")
print(f"Reduced-group directive (None means: skip the model, use "
      f"FIXED_SENTENCES['reduced'] instead): {example_reduced_directive}")
if example_increased_directive is not None:
    print(f"\nExample [USER] message the model would actually see for the 'increased' "
          f"group of this claim:\n\n{user_prompt_template.format(directives=example_increased_directive)}")

### Why this generalizes to unseen data

The skeleton isn't a lookup table and isn't fit on training data - `build_skeleton()`
reads `direction` straight off *this claim's own* SHAP values, which are computed
fresh at inference time for whatever claim comes in. A brand-new claim the model has
never seen still gets its own SHAP explanation, and that explanation's sign is what
`split_drivers_by_direction()` sorts on - there is no case where a feature's direction
is unknown or has to be guessed, because it's a property of that claim's own
prediction, not something remembered from other claims.

Because the sorting step is ordinary code (an `if`/`else` on a sign, not a model call),
it is deterministic and correct by construction for every claim, seen or unseen - it
cannot "have an off day" the way a 3B-parameter model reading a direction label can.
Qwen only ever receives the two already-sorted groups and is only asked to turn them
into prose; it never sees an unsorted list and is never in a position to reassign a
feature, so a direction flip is not a mistake it can make on this claim or the next
one - the failure mode is removed from the pipeline, not just made less likely.

## 4. Step 2 - Generate Briefs

**Structural fix (kept): the model never sees the fallback text.** An earlier version
of this notebook sent one flat prompt per claim showing the model *both* branches of
the increased/reduced conditional, including the literal fallback sentence ("No
factors increased/reduced the fraud risk") for whichever branch didn't apply - Qwen
repeatedly echoed that fallback text even when the corresponding driver list was
genuinely non-empty. The fix: code already knows with certainty whether each driver
group is empty (`build_group_directive()`). An empty group's sentence comes from
`FIXED_SENTENCES`, spliced in directly - **zero LLM involvement, zero risk**. A
non-empty group is the *only* thing a given Ollama call is ever asked to phrase, and
that call never sees the other group's data or either fixed sentence. Each claim gets
0, 1, or 2 Ollama calls, and the final brief is assembled by code from a code-composed
lead-in sentence, the increased-risk sentence, the reduced-risk sentence, and a
code-composed action sentence.

**Sampling (revised): greedy decoding, not Qwen's general-purpose recommended
defaults.** An earlier revision moved this notebook to `temperature=0.7/top_p=0.8/
top_k=20/repeat_penalty=1.1`, reasoning that Qwen's own shipped defaults would improve
instruction-following generally. In practice, on the exact task each call now does -
"restate these 1-5 given facts plainly, add nothing" - that backfired: two new failure
patterns appeared (dismissing a correctly-listed factor as having "no impact," and
inventing ungrounded rationalizations for why a factor moved the risk) on claims that
had rendered cleanly before. The mechanism: `repeat_penalty` discourages the model from
literally repeating tokens it just saw in its own (short) prompt, nudging it toward
paraphrase and invented elaboration instead of verbatim restatement - exactly backwards
for a task that *wants* verbatim restatement. Creative sampling was only ever needed to
fix the fallback-anchoring bug above, and that fix no longer depends on temperature at
all - so sampling is back to `temperature=0, repeat_penalty=1.0` (see `GENERATION_OPTIONS`),
chosen purely for this task's literalness requirement rather than general-purpose advice.

Calls use Ollama's `/api/chat` (separate `system`/`user` roles, matching how Qwen2.5 was
instruction-tuned) rather than `/api/generate` with one undifferentiated prompt string.
Each call still gets one retry on a *transport* failure (timeout, malformed response)
before being recorded as a failure.

`detect_boilerplate_contradiction()` still runs once per claim, on the final assembled
brief, as a passive safety net (no retry) - with the fallback text structurally absent
from every prompt, it should have nothing left to catch, but any claim it does flag is
still surfaced in Step 3's review rather than silently accepted.

In [48]:
def call_ollama_chat(system_text: str, user_text: str, base_url: str, model_name: str, options: dict) -> str:
    """/api/chat, not /api/generate: separate system/user roles match how Qwen2.5 was
    instruction-tuned (ChatML), rather than relying on the model to infer a split from
    one undifferentiated prompt string. Response shape differs from /api/generate too -
    the text is at payload['message']['content'], not payload['response']."""
    response = requests.post(
        f"{base_url}/api/chat",
        json={
            "model": model_name,
            "messages": [
                {"role": "system", "content": system_text},
                {"role": "user", "content": user_text},
            ],
            "stream": False,
            "options": options,
        },
        timeout=180,
    )
    response.raise_for_status()
    payload = response.json()
    if "message" not in payload or "content" not in payload["message"]:
        raise ValueError(f"Ollama /api/chat response had no message.content field: {payload}")
    return payload["message"]["content"]


def generate_with_retry(system_text: str, user_text: str, base_url: str, model_name: str,
                          options: dict, max_retries: int) -> dict:
    """One call, one retry on any transport failure - never raises, always returns a
    status dict so a single bad claim can't stop the run. Content-level correctness
    (whether the response contradicts the skeleton) is checked separately, after this
    returns - retrying a bad transport call and retrying a bad *answer* are different
    problems with different fixes."""
    last_error = None
    for attempt in range(1, max_retries + 2):
        try:
            raw_response = call_ollama_chat(system_text, user_text, base_url, model_name, options)
            return {"status": "ok", "raw_response": raw_response, "attempts": attempt, "error": None}
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            print(f"    attempt {attempt} failed: {last_error}")
            if attempt <= max_retries:
                time.sleep(2)
    return {"status": "failed", "raw_response": None, "attempts": max_retries + 1, "error": last_error}


BRIEF_PATTERN = re.compile(r"BRIEF:\s*(.*)", re.IGNORECASE | re.DOTALL)


def parse_brief_response(raw_response: str) -> dict:
    """Pull the one-sentence text out of the model's response. Falls back to the full
    raw text if the model didn't follow the requested 'BRIEF: ...' format - flagged via
    parse_ok, not silently accepted. No ACTION line is requested anymore: the action was
    already fixed by risk tier in build_skeleton() and never depended on what the model
    said, so asking it to echo one back had no effect and is no longer asked for."""
    brief_match = BRIEF_PATTERN.search(raw_response)
    sentence_text = brief_match.group(1).strip() if brief_match else raw_response.strip()
    return {"sentence_text": sentence_text, "parse_ok": bool(brief_match)}


print("Generation harness ready: call_ollama_chat, generate_with_retry, parse_brief_response.")

Generation harness ready: call_ollama_chat, generate_with_retry, parse_brief_response.


In [49]:
def detect_boilerplate_contradiction(brief_text: str, skeleton: dict) -> list:
    """Ground-truth check against the skeleton THIS brief was built from - not a guess,
    and not dependent on brief wording. With the structural fix above (the model is
    never shown a fixed sentence or the other group's data), this specific
    contradiction should no longer be reachable - kept as a permanent, free tripwire in
    Step 3's review flags rather than removed outright, in case some other path
    (a hand-edited template, a future prompt change) reintroduces it."""
    if not brief_text:
        return []
    text_lower = brief_text.lower()
    contradictions = []
    if skeleton["increased_fraud_risk"] and FIXED_SENTENCES["increased"].lower() in text_lower:
        contradictions.append(
            f"says '{FIXED_SENTENCES['increased']}' but "
            f"{len(skeleton['increased_fraud_risk'])} were given in the skeleton"
        )
    if skeleton["reduced_fraud_risk"] and FIXED_SENTENCES["reduced"].lower() in text_lower:
        contradictions.append(
            f"says '{FIXED_SENTENCES['reduced']}' but "
            f"{len(skeleton['reduced_fraud_risk'])} were given in the skeleton"
        )
    return contradictions


print("Content-validation check ready: detect_boilerplate_contradiction (review-time flag, no retry).")

Content-validation check ready: detect_boilerplate_contradiction (review-time flag, no retry).


In [50]:
generated_briefs = []
n_still_contradicting = 0
for i, claim in enumerate(claims, start=1):
    claim_id = claim["claim_id"]
    print(f"[{i}/{len(claims)}] {claim_id} ...", end=" ")
    skeleton = build_skeleton(claim)
    lead_in = compose_lead_in_sentence(skeleton)
    action_sentence = compose_action_sentence(skeleton)

    sentences = {}
    group_calls = []
    claim_status = "ok"
    for direction in ("increased", "reduced"):
        drivers = skeleton[f"{direction}_fraud_risk"]
        directive = build_group_directive(direction, drivers)

        if directive is None:
            # Empty group: the fixed sentence is spliced in directly - zero LLM
            # involvement, zero risk. No call is made, nothing is recorded in
            # group_calls, because nothing was asked of the model for this group.
            sentences[direction] = FIXED_SENTENCES[direction]
            continue

        user_text = user_prompt_template.format(directives=directive)
        assert_no_fallback_leakage(system_prompt_template, user_text)
        result = generate_with_retry(system_prompt_template, user_text, OLLAMA_BASE_URL,
                                      MODEL_NAME, GENERATION_OPTIONS, MAX_RETRIES)

        if result["status"] == "ok":
            parsed = parse_brief_response(result["raw_response"])
            sentences[direction] = parsed["sentence_text"]
            group_calls.append({
                "direction": direction, "status": "ok", "attempts": result["attempts"],
                "error": None, "raw_response": result["raw_response"],
                "parse_ok": parsed["parse_ok"], "user_message": user_text,
            })
        else:
            sentences[direction] = None
            claim_status = "failed"
            group_calls.append({
                "direction": direction, "status": "failed", "attempts": result["attempts"],
                "error": result["error"], "raw_response": None,
                "parse_ok": False, "user_message": user_text,
            })

    if claim_status == "ok":
        brief = " ".join([lead_in, sentences["increased"], sentences["reduced"], action_sentence])
    else:
        brief = None
    contradictions = detect_boilerplate_contradiction(brief, skeleton) if brief else []
    n_still_contradicting += bool(contradictions)

    n_calls = len(group_calls)
    parse_ok = all(gc["parse_ok"] for gc in group_calls) if group_calls else True
    print(f"{claim_status} ({n_calls} LLM call(s), parse_ok={parse_ok}"
          f"{', CONTRADICTS SKELETON' if contradictions else ''})")

    generated_briefs.append({
        "claim_id": claim_id,
        "status": claim_status,
        "brief": brief,
        "recommended_action": skeleton["action"] if claim_status == "ok" else None,
        "parse_ok": parse_ok,
        "boilerplate_contradictions": contradictions,
        # Exactly which 0/1/2 Ollama calls happened for this claim, and what each one
        # was asked (user_message never contains the other group's data or any fixed
        # sentence - see assert_no_fallback_leakage above).
        "group_calls": group_calls,
        # Evidence for the report: the exact code-built skeleton this brief was
        # rendered from, alongside the model's own prose - so a reviewer can compare
        # them directly and see the directions were assigned by code, not by Qwen.
        "skeleton": skeleton,
        "inputs_used": {
            "risk_tier": claim["risk_tier"],
            "calibrated_fraud_probability": claim["calibrated_fraud_probability"],
            "true_label": claim["true_label"],
            "prediction_outcome": claim["prediction_outcome"],
            # Trimmed to what was actually shown to the model (top TOP_N_DRIVERS_FOR_PROMPT
            # by |SHAP|), not the full claim["top_shap_drivers"] - so this record, and
            # anything downstream that reads it, can't treat a dropped near-zero driver as
            # something the model saw.
            "top_shap_drivers": select_top_drivers(claim["top_shap_drivers"], TOP_N_DRIVERS_FOR_PROMPT),
        },
    })

n_ok = sum(b["status"] == "ok" for b in generated_briefs)
n_failed = len(generated_briefs) - n_ok
n_parse_issues = sum(b["status"] == "ok" and not b["parse_ok"] for b in generated_briefs)
n_calls_total = sum(len(b["group_calls"]) for b in generated_briefs)
print(f"\nDone: {n_ok} generated, {n_failed} failed after retry, "
      f"{n_parse_issues} generated but didn't match the requested BRIEF format "
      "(the raw response was kept as that group's sentence either way).")
print(f"LLM calls made: {n_calls_total} across {len(claims)} claims (up to 2 per claim - "
      "an empty group needs zero calls, its sentence comes from FIXED_SENTENCES instead).")
print(f"Boilerplate-contradiction check: {n_still_contradicting} of {n_ok} successful briefs "
      "still contradicted their own skeleton despite the structural fix - these are "
      "flagged in boilerplate_contradictions and surfaced in Step 3's review, never "
      "silently accepted. Expected to be 0 or near-0 now that the model is never shown "
      "the fallback text for a non-empty group.")

[1/46] TEST-12437 ... ok (2 LLM call(s), parse_ok=False)
[2/46] TEST-04027 ... ok (1 LLM call(s), parse_ok=False)
[3/46] TEST-07330 ... ok (2 LLM call(s), parse_ok=False)
[4/46] TEST-13495 ... ok (2 LLM call(s), parse_ok=False)
[5/46] TEST-06854 ... ok (1 LLM call(s), parse_ok=False)
[6/46] TEST-09945 ... ok (2 LLM call(s), parse_ok=False)
[7/46] TEST-01682 ... ok (1 LLM call(s), parse_ok=False)
[8/46] TEST-01945 ... ok (2 LLM call(s), parse_ok=False)
[9/46] TEST-04807 ... ok (1 LLM call(s), parse_ok=False)
[10/46] TEST-00111 ... ok (1 LLM call(s), parse_ok=False)
[11/46] TEST-09299 ... ok (2 LLM call(s), parse_ok=False)
[12/46] TEST-05376 ... ok (2 LLM call(s), parse_ok=False)
[13/46] TEST-07018 ... ok (2 LLM call(s), parse_ok=False)
[14/46] TEST-00757 ... ok (1 LLM call(s), parse_ok=False)
[15/46] TEST-03884 ... ok (2 LLM call(s), parse_ok=False)
[16/46] TEST-06183 ... ok (2 LLM call(s), parse_ok=False)
[17/46] TEST-05429 ... ok (1 LLM call(s), parse_ok=False)
[18/46] TEST-07598 ... 

### Save generated briefs

In [51]:
GENERATED_BRIEFS_PATH.parent.mkdir(parents=True, exist_ok=True)
generated_briefs_export = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model": MODEL_NAME,
    "generation_options": GENERATION_OPTIONS,  # constant across the whole run - see Section 1
    "source_export": str(SHAP_EXPORT_PATH),
    "n_claims": len(generated_briefs),
    "n_ok": n_ok,
    "n_failed": n_failed,
    "n_still_contradicting": n_still_contradicting,
    "briefs": generated_briefs,
}
with open(GENERATED_BRIEFS_PATH, "w", encoding="utf-8") as f:
    json.dump(generated_briefs_export, f, indent=2)
print(f"Saved -> {GENERATED_BRIEFS_PATH.resolve()}  ({len(generated_briefs)} briefs)")

Saved -> C:\Users\LENOVO\Desktop\samsung-project\insurance-fraud-detection\notebooks\results\generated_briefs.json  (46 briefs)


## 5. Step 3 - Rubric Evaluation Scaffold

This step does not score anything - it builds the table a human scores by
hand. Each criterion is 0/1/2:

| Criterion | 0 | 1 | 2 |
|---|---|---|---|
| `factual_consistency` | Brief contradicts a given feature value or driver | Mostly consistent, one minor inaccuracy | Accurately reflects every referenced value/driver |
| `faithfulness_to_shap` | Stated reasons don't match the actual top SHAP drivers | Mentions some but not the most important driver(s), or gets a direction wrong | Explanation matches the top drivers and their direction |
| `actionability` | Recommended action missing, unclear, or inconsistent with the risk tier | Present but only loosely justified | Clear, present, and well-justified given the tier |
| `no_fabrication` | Invents claim details, amounts, names, or evidence not given | Vague/generic language that isn't clearly fabricated but isn't grounded either | Strictly uses only the given feature values and drivers |

**Gate rule** (documented here, applied in Step 4 once scores exist): a
brief is acceptable only if its **total score is >= 6/8 AND it scores
exactly 2 on both `factual_consistency` and `no_fabrication`** - a brief
that is merely well-written but ungrounded, or grounded but useless, does
not pass.

**Automated fabrication pre-check:** a cheap heuristic, not a replacement
for the `no_fabrication` column above - it flags (1) any dollar amount
(this dataset has no claim-amount field, so any `$` figure is fabricated by
construction), and (2) any mention of a real value from this claim's *other*
~20 raw columns that were never given to the model (only this claim's SHAP
drivers were). It is scoped to what was actually in the prompt, so it can
miss subtler fabrications (a plausible-sounding but entirely invented
detail) and can occasionally flag ordinary language that happens to overlap
with an unrelated real value - a pointer for the human reviewer, not a
verdict.

In [52]:
def detect_possible_fabrication(brief_text: str, claim: dict, prompt_drivers: list) -> list:
    """Cheap heuristics to guide (not replace) human review - see the markdown above
    for exactly what this does and does not catch. `prompt_drivers` must be the exact,
    already-trimmed driver list actually shown to the model (see inputs_used /
    TOP_N_DRIVERS_FOR_PROMPT) - not claim["top_shap_drivers"] - so a near-zero driver
    dropped before generation is correctly treated as never given to the model, not as
    an allowed feature the brief happens not to mention."""
    if not brief_text:
        return []
    reasons = []
    if re.search(r"\$\s?\d", brief_text):
        reasons.append("mentions a dollar amount (this dataset has no claim-amount field)")

    driver_features = {d["feature"] for d in prompt_drivers}
    text_lower = brief_text.lower()
    for feature, value in claim["original_feature_values"].items():
        if feature in driver_features:
            continue
        if isinstance(value, str) and len(value) >= 4 and value.lower() in text_lower:
            reasons.append(f"mentions '{value}' ({feature}) - not one of the drivers given to the model")
    return reasons


def format_drivers_for_review(drivers: list) -> str:
    return "; ".join(f"{d['feature']}={d['value']} ({d['direction']}, {get_shap_contribution(d):+.3f})"
                      for d in drivers)


REVIEW_SAMPLE_SIZE = None  # None = review every generated brief; set an int to review a subset
briefs_to_review = generated_briefs if REVIEW_SAMPLE_SIZE is None else generated_briefs[:REVIEW_SAMPLE_SIZE]
print(f"Reviewing {len(briefs_to_review)} of {len(generated_briefs)} generated briefs.")

review_rows = []
for b in briefs_to_review:
    claim = claims_by_id[b["claim_id"]]
    prompt_drivers = b["inputs_used"]["top_shap_drivers"]  # exactly what this brief's prompt showed the model
    # Combines the fabrication heuristic above with the boilerplate-contradiction check
    # already run (and retried against) during generation - saved on the brief itself,
    # not recomputed here, so this reflects the same skeleton the brief was built from.
    fabrication_flags = detect_possible_fabrication(b["brief"], claim, prompt_drivers)
    fabrication_flags = fabrication_flags + b.get("boilerplate_contradictions", [])
    review_rows.append({
        "claim_id": b["claim_id"],
        "risk_tier": claim["risk_tier"],
        "calibrated_fraud_probability": claim["calibrated_fraud_probability"],
        "true_label": claim["true_label"],
        "prediction_outcome": claim["prediction_outcome"],
        "shap_drivers": format_drivers_for_review(prompt_drivers),
        "generated_brief": b["brief"],
        "recommended_action": b["recommended_action"],
        "generation_status": b["status"],
        "automated_fabrication_flags": "; ".join(fabrication_flags) if fabrication_flags else "",
        "factual_consistency": "",
        "faithfulness_to_shap": "",
        "actionability": "",
        "no_fabrication": "",
        "reviewer_notes": "",
    })

review_df = pd.DataFrame(review_rows)
n_flagged = int((review_df["automated_fabrication_flags"] != "").sum())
print(f"Automated pre-check flagged {n_flagged} of {len(review_df)} briefs for a possible fabrication - "
      "review these first, but don't skip the rest.")
display(review_df[["claim_id", "generation_status", "automated_fabrication_flags"]])

Reviewing 46 of 46 generated briefs.
Automated pre-check flagged 2 of 46 briefs for a possible fabrication - review these first, but don't skip the rest.


,claim_id,generation_status,automated_fabrication_flags
0,TEST-12437,ok,
1,TEST-04027,ok,
2,TEST-07330,ok,
3,TEST-13495,ok,
4,TEST-06854,ok,
5,TEST-09945,ok,
6,TEST-01682,ok,
7,TEST-01945,ok,
8,TEST-04807,ok,
9,TEST-00111,ok,


### Save the rubric scaffold

In [53]:
RUBRIC_SCORES_PATH.parent.mkdir(parents=True, exist_ok=True)
review_df.to_csv(RUBRIC_SCORES_PATH, index=False)
print(f"Saved -> {RUBRIC_SCORES_PATH.resolve()}  ({len(review_df)} rows)")

Saved -> C:\Users\LENOVO\Desktop\samsung-project\insurance-fraud-detection\notebooks\rubric\brief_scores.csv  (46 rows)


### Lightweight sanity checks

This notebook has no frozen model artifacts to protect, but it does have
one integrity guarantee worth proving: every claim that went in comes back
out, exactly once, all the way through to the rubric file.

In [54]:
assert set(b["claim_id"] for b in generated_briefs) == set(claims_by_id.keys()), (
    "generated_briefs' claim_ids do not exactly match the input export's claim_ids - "
    "a claim was dropped, duplicated, or renamed somewhere in this notebook."
)
assert len(generated_briefs) == len(set(b["claim_id"] for b in generated_briefs)), (
    "Duplicate claim_id found in generated_briefs."
)
print(f"PASSED: all {len(claims)} claim IDs round-trip exactly between the input export "
      "and the generated briefs, with no duplicates.")

reloaded_review_df = pd.read_csv(RUBRIC_SCORES_PATH)
assert len(reloaded_review_df) == len(briefs_to_review), "Rubric CSV row count does not match the reviewed briefs."
assert set(RUBRIC_CRITERIA).issubset(reloaded_review_df.columns), "Rubric CSV is missing one or more score columns."
print(f"PASSED: {RUBRIC_SCORES_PATH.name} has exactly one row per reviewed claim and all "
      "four rubric score columns (currently blank, awaiting human scoring).")

PASSED: all 46 claim IDs round-trip exactly between the input export and the generated briefs, with no duplicates.
PASSED: brief_scores.csv has exactly one row per reviewed claim and all four rubric score columns (currently blank, awaiting human scoring).


### Human scoring gate - stop here

**This is a manual step.** Open `rubric/brief_scores.csv` (Excel, Google
Sheets, or any CSV-aware editor) and fill in `factual_consistency`,
`faithfulness_to_shap`, `actionability`, and `no_fabrication` for each row,
using the 0/1/2 definitions above. The `automated_fabrication_flags` column
is a hint, not an answer - read the brief itself before scoring
`no_fabrication`.

Step 4 below reads the CSV back from disk, so it picks up whatever has been
filled in - it works whether scoring is complete, partial, or (harmlessly)
not yet started.

## 6. Step 4 - Summary for the Report

Re-reads `rubric/brief_scores.csv` from disk - not the in-memory
`review_df` above - since the whole point of Step 3 was to hand it off to a
human editing the file directly. Run this cell any time after scoring
starts; it reports how much of the file is actually scored rather than
assuming it's complete.

In [58]:
scored_df = pd.read_csv(RUBRIC_SCORES_PATH)

is_unscored = scored_df[RUBRIC_CRITERIA].isna() | (scored_df[RUBRIC_CRITERIA].astype(str).apply(lambda c: c.str.strip()) == "")
any_unscored = is_unscored.any(axis=1)
n_unscored = int(any_unscored.sum())

if n_unscored == len(scored_df):
    print(f"No briefs have been scored yet in {RUBRIC_SCORES_PATH} - fill in at least "
          "some rows, then re-run this cell. Nothing else below will have anything to show.")
    scored = scored_df.iloc[0:0].copy()
else:
    if n_unscored > 0:
        print(f"NOTE: {n_unscored} of {len(scored_df)} briefs are not yet fully scored - "
              f"summary stats below cover only the {len(scored_df) - n_unscored} that are.")
    scored = scored_df.loc[~any_unscored].copy()
    for col in RUBRIC_CRITERIA:
        scored[col] = pd.to_numeric(scored[col], errors="raise")

if len(scored) > 0:
    scored["total_score"] = scored[RUBRIC_CRITERIA].sum(axis=1)
    scored["passes_gate"] = (
        (scored["total_score"] >= 6)
        & (scored["factual_consistency"] == 2)
        & (scored["no_fabrication"] == 2)
    )

    pass_rate = 100 * scored["passes_gate"].mean()
    mean_by_criterion = scored[RUBRIC_CRITERIA].mean().to_frame("mean_score")

    print(f"\nScored: {len(scored)} / {len(scored_df)} briefs")
    print(f"Gate pass rate: {pass_rate:.1f}%  "
          "(total >= 6/8 AND factual_consistency == 2 AND no_fabrication == 2)")
    display(mean_by_criterion)
else:
    pass_rate = None
    mean_by_criterion = None


Scored: 46 / 46 briefs
Gate pass rate: 95.7%  (total >= 6/8 AND factual_consistency == 2 AND no_fabrication == 2)


,mean_score
factual_consistency,1.978261
faithfulness_to_shap,1.869565
actionability,2.000000
no_fabrication,1.978261


### Example briefs for the report

In [59]:
if len(scored) == 0:
    print("No scored briefs yet - nothing to surface. Re-run this cell after scoring.")
else:
    strong_examples = scored.loc[scored["passes_gate"]].sort_values("total_score", ascending=False).head(1)
    flagged_examples = scored.loc[~scored["passes_gate"]].sort_values("total_score", ascending=True).head(2)
    examples = pd.concat([strong_examples, flagged_examples])

    if examples.empty:
        print("No examples to show yet (need at least one scored row).")
    for _, row in examples.iterrows():
        label = "STRONG (passes gate)" if row["passes_gate"] else "FLAGGED (fails gate)"
        print(f"\n{'=' * 78}\n[{label}] Claim {row['claim_id']}  |  "
              f"total={row['total_score']:.0f}/8  |  risk_tier={row['risk_tier']}  |  "
              f"probability={row['calibrated_fraud_probability']}")
        print(f"SHAP drivers: {row['shap_drivers']}")
        print(f"Brief: {row['generated_brief']}")
        print(f"Recommended action: {row['recommended_action']}")
        # pd.read_csv turns an empty string back into NaN, and bool(nan) is True in
        # Python - checking truthiness directly here would print "nan" for every claim.
        if pd.notna(row["automated_fabrication_flags"]) and str(row["automated_fabrication_flags"]).strip():
            print(f"Automated fabrication flag: {row['automated_fabrication_flags']}")


[STRONG (passes gate)] Claim TEST-12437  |  total=8/8  |  risk_tier=High  |  probability=0.1743
SHAP drivers: BasePolicy=Collision (up, +0.357); Fault=Policy Holder (up, +0.264); Year=1996 (down, -0.164); VehiclePrice=more than 69000 (up, +0.072); AgeOfVehicle=new (down, -0.054)
Brief: This claim is High risk, with a calibrated fraud probability of 0.17. The fraud risk increased due to BasePolicy being Collision, Fault being Policy Holder, and VehiclePrice being more than 69000. The factors Year = 1996 and AgeOfVehicle = new lowered the fraud risk. This claim is routed for Fraud Investigation.
Recommended action: Fraud Investigation

[FLAGGED (fails gate)] Claim TEST-06928  |  total=6/8  |  risk_tier=Low  |  probability=0.0735
SHAP drivers: Fault=Policy Holder (up, +0.240); DayOfWeek=Tuesday (down, -0.145); Age=53.0 (down, -0.115); NumberOfSuppliments=more than 5 (down, -0.095); Make=Pontiac (down, -0.077)
Brief: This claim is Low risk, with a calibrated fraud probability of 0.07. Fau

## Qwen Briefs & Rubric Evaluation Summary

In [60]:
summary_lines = [
    f"**Briefs generated:** {n_ok} / {len(claims)} succeeded ({n_failed} failed after one "
    f"transport retry, {n_parse_issues} generated but one of their sentences didn't match "
    "the requested BRIEF format - the raw response was kept either way).",
    f"**Model:** `{MODEL_NAME}` via local Ollama `/api/chat` (system/user roles), "
    f"options {GENERATION_OPTIONS} - deliberately greedy decoding, not Qwen's general-"
    "purpose creative-sampling defaults, which were found to nudge the model toward "
    "paraphrase/invented rationalization on this literal-restatement task (see Section 4).",
    "**Direction assignment:** done entirely in code (`split_drivers_by_direction`), from "
    "each claim's own SHAP `direction` field, before Qwen ever sees the claim - Qwen only "
    "renders one already-sorted group at a time into one sentence and cannot flip one.",
    "**Action assignment:** done entirely in code (`derive_action_from_tier`) and never "
    "asked of the model at all - there is nothing left for Qwen to echo or override.",
    "**Structural fix for the boilerplate-contradiction bug:** code, not the model, "
    "decides whether each driver group is empty. An empty group's sentence "
    "(`FIXED_SENTENCES`) is spliced in directly with zero LLM involvement; a non-empty "
    "group is the only thing a given LLM call is ever asked to phrase, and it is never "
    f"shown the other group's data or either fixed sentence ({n_calls_total} total LLM "
    f"calls across {len(claims)} claims - up to 2 per claim, fewer wherever a group was "
    "empty). `detect_boilerplate_contradiction` still runs as a passive, non-retrying "
    f"safety net: {n_still_contradicting} of {n_ok} successful briefs contradicted their "
    "own skeleton despite this, and are flagged in the rubric's automated_fabrication_flags "
    "column for human review rather than silently accepted.",
    f"**Rubric scaffold:** {len(review_df)} briefs written to `{RUBRIC_SCORES_PATH}` for human "
    f"scoring, {n_flagged} pre-flagged by the automated fabrication check.",
]
if pass_rate is not None:
    summary_lines.append(
        f"**Human scoring (as of this run):** {len(scored)} / {len(scored_df)} briefs scored, "
        f"{pass_rate:.1f}% pass the gate (>=6/8 total, factual_consistency==2, no_fabrication==2)."
    )
else:
    summary_lines.append(
        "**Human scoring:** not started yet - re-run Step 4 after filling in "
        f"`{RUBRIC_SCORES_PATH}`."
    )
summary_lines.append(
    "**Nothing here calls a cloud API, retrains anything, or fabricates a rubric score** - "
    "every score in the CSV is either blank (unscored) or was typed in by a human."
)
print("\n\n".join(summary_lines))

**Briefs generated:** 46 / 46 succeeded (0 failed after one transport retry, 46 generated but one of their sentences didn't match the requested BRIEF format - the raw response was kept either way).

**Model:** `qwen2.5:3b` via local Ollama `/api/chat` (system/user roles), options {'temperature': 0, 'repeat_penalty': 1.0, 'seed': 42} - deliberately greedy decoding, not Qwen's general-purpose creative-sampling defaults, which were found to nudge the model toward paraphrase/invented rationalization on this literal-restatement task (see Section 4).

**Direction assignment:** done entirely in code (`split_drivers_by_direction`), from each claim's own SHAP `direction` field, before Qwen ever sees the claim - Qwen only renders one already-sorted group at a time into one sentence and cannot flip one.

**Action assignment:** done entirely in code (`derive_action_from_tier`) and never asked of the model at all - there is nothing left for Qwen to echo or override.

**Structural fix for the boiler

### Files saved by this notebook

In [61]:
print("Qwen briefs/rubric outputs written this run:")
for p in [GENERATED_BRIEFS_PATH, RUBRIC_SCORES_PATH]:
    print(f"  - {p.resolve()}")
print(f"\n(Not committed: {PROMPT_TEMPLATE_PATH} - git-ignored by design; see Section 3.)")

Qwen briefs/rubric outputs written this run:
  - C:\Users\LENOVO\Desktop\samsung-project\insurance-fraud-detection\notebooks\results\generated_briefs.json
  - C:\Users\LENOVO\Desktop\samsung-project\insurance-fraud-detection\notebooks\rubric\brief_scores.csv

(Not committed: prompts\brief_prompt.txt - git-ignored by design; see Section 3.)
